# Solutions · Chapter 02-05 · Distributions, outliers, and transformations

Worked answers with reasoning. E4 shows an outlier hiding itself from the rule meant to catch it,
and E14 shows that keeping the rows without the flag is nearly as bad as deleting them - which
reframes the whole chapter.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(17)
n_days = 900
temperature = rng.normal(18, 6, n_days)
festival = rng.random(n_days) < 0.04
rentals = np.clip(40 + 3.5 * temperature + rng.normal(0, 12, n_days) + festival * 260, 5, None).round()
days = pd.DataFrame({"temp_c": temperature.round(1), "festival": festival.astype(int),
                     "rentals": rentals})
train, test = np.arange(n_days) < 700, np.arange(n_days) >= 700
is_festival_test = festival[test]
print(f"{n_days} days, {festival.sum()} festivals, mean {rentals.mean():.1f}, median {np.median(rentals):.1f}")

## E1 · The three kinds of outlier

| Kind | What it is | Response |
|---|---|---|
| **An error** | The value is wrong - a typo, a unit slip, a sentinel | Correct it if you can, otherwise set it to missing and handle it as 02-04 describes. Never leave it silently in |
| **Rare but real** | The value is right and unusual | **Keep it.** It is evidence. Use a robust method or a transform if its influence is a problem |
| **A different population** | The row was produced by a different process | **Keep it and label it.** Add a feature identifying the group, or model the groups separately |

**The order matters.** Ask "is it possible?" first, because that is answerable from domain knowledge
alone. Only then ask "is it a different kind of thing?".

## E2 · Why three-sigma took exactly the festivals

Because "far from the mean" and "belongs to the second population" are the same set of rows when one
population sits far above the other.

The mean of 110.8 and the standard deviation of 50.8 are both computed over the **mixture**. The
ordinary days cluster near 100 with a spread of about 20; the festival days sit near 360. A cut-off
at three standard deviations lands at 263 - above every ordinary day and below every festival day.
The rule became a perfect classifier for "is this a festival", and then deleted that class.

**The general statement:** a distance-from-the-centre rule cannot distinguish "unusual within this
population" from "belongs to a different population", because both look identical from the centre.
Nothing about the distance tells you which - only knowing what the rows *are* does.

**And a nasty second-order effect:** the festival days inflate the standard deviation, which widens
the cut-off, which makes the rule *less* likely to flag them. E4 shows a case where that goes far
enough to hide the outlier completely.

## E3 · Mean 110.8, median 104.0

**The mean answers:** if I pooled every rental across all 900 days and shared them equally, each day
would get 110.8. It is a statement about the total.

**The median answers:** put the days in order and take the middle one - half of days were busier,
half quieter. It is a statement about a typical case.

**For Maria's question - "what does a normal day look like?" - the median, 104.**

The mean is above about 60% of days, pulled up by 29 festivals out of 900. A number that overstates
five days in eight is a poor description of a normal day.

**But note the mean is the right answer to a different question she will also ask.** "How many bikes
do I need to have available across the year?" is about the total, and there the mean is exactly
right - the festivals really did happen and really did consume bikes. **Neither summary is better;
they answer different questions**, and the professional move is to say which one you are answering.

## E4 · An outlier that hides from the rule meant to catch it

In [ ]:
seven = np.array([90, 95, 100, 105, 110, 115, 400], dtype=float)
without = seven[:-1]

print(f"mean with the 400      : {seven.mean():.1f}")
print(f"median                 : {np.median(seven):.1f}")
print(f"mean without the 400   : {without.mean():.1f}")
print()
print(f"standard deviation with the 400   : {seven.std(ddof=1):.1f}")
print(f"standard deviation without it     : {without.std(ddof=1):.1f}")

limit = seven.mean() + 3 * seven.std(ddof=1)
print(f"\nthree-sigma upper cut-off: {limit:.0f}")
print(f"is 400 flagged as an outlier? {400 > limit}")

| | With the 400 | Without it |
|---|---|---|
| Mean | 145.0 | 102.5 |
| Standard deviation | **112.8** | **9.4** |

One value in seven **multiplied the standard deviation by twelve**.

**And the punchline: the three-sigma cut-off is 483, so 400 is not flagged at all.** The rule
designed to catch the outlier fails to catch it, because the outlier itself inflated the yardstick.

This has a name - **masking** - and it is a general property of any threshold built from
non-robust statistics. The more extreme the point, the more it widens the interval meant to exclude
it. With two or three such points it gets worse: they conceal each other.

**Two consequences worth carrying:**

1. **If you must use a threshold, build it from robust statistics** - the median and the
   interquartile range, or plain quantiles - which the extremes cannot inflate. The common form is
   the IQR rule: flag beyond `Q1 - 1.5*IQR` or `Q3 + 1.5*IQR`.
2. **The chapter's failure and this one are opposite errors from the same cause.** On 900 days the
   rule deleted an entire real population; on seven days it missed a genuine extreme. Both because
   the standard deviation is not a stable ruler when the data has heavy tails.

## E5 · Doubling, and what logs do to it

In [ ]:
doubling = np.array([5, 10, 20, 40, 80, 160], dtype=float)
print("values          :", doubling)
print("differences     :", np.diff(doubling))
print("log2 values     :", np.log2(doubling))
print("log2 differences:", np.diff(np.log2(doubling)))

The raw differences are 5, 10, 20, 40, 80 - growing, because each step adds more than the last. The
**log2 differences are 1, 1, 1, 1, 1** - constant, because each step is the same *multiplication*.

**What the transform did:** it converted a constant ratio into a constant distance. Equal
multiplicative steps become equal additive steps.

**What a straight line on log axes means:** a **constant growth rate** - the quantity is multiplied
by the same factor each period. The slope of that line *is* the growth rate: on a log2 axis, a slope
of 1 per step means doubling every step.

**Why this matters for modelling:** a straight line is what linear models fit. If a relationship is
exponential in the raw data, no linear model will capture it - but the same model on logged data
fits it exactly. **Choosing a transform is choosing which relationships your model can express**,
which is chapter 05-07's subject.

**And the diagnostic in one sentence:** curved on raw axes, straight on log axes means
multiplicative. Straight on both means additive. Curved on both means neither, and you need
something else.

## E6 · A distribution report

In [ ]:
def distribution_report(series, name="", mode_decimals=-1):
    """mode_decimals sets the rounding used for the mode: -1 for counts, 2 for logged values."""
    s = series.dropna()
    mean, sd = s.mean(), s.std()
    beyond_sigma = ((s - mean).abs() > 3 * sd).mean()
    print(f"--- {name or series.name} ---")
    print(f"count {len(s):>6}   mean {mean:9.2f}   median {s.median():9.2f}   "
          f"mode {s.round(mode_decimals).mode().iloc[0]:9.2f}")
    print(f"skew  {s.skew():6.2f}   sd   {sd:9.2f}")
    print(f"quartiles  {s.quantile(0.25):.2f} | {s.median():.2f} | {s.quantile(0.75):.2f}")
    print(f"1st pct {s.quantile(0.01):.2f}   99th pct {s.quantile(0.99):.2f}")
    print(f"beyond 3 sd: {beyond_sigma:.2%}   above the 99th pct: {(s > s.quantile(0.99)).mean():.2%}")


distribution_report(days["rentals"], "rentals, as measured")
print()
distribution_report(np.log10(days["rentals"]), "log10(rentals)", mode_decimals=2)

The line worth staring at is **"beyond 3 sd"**.

On the raw data it is **3.22%** - more than ten times the 0.3% the rule assumes. That single number
is the whole warning: if the three-sigma rule is flagging 3% of your rows, your data is not normal
and the rule is not doing what its justification says it does.

**And the logged data is no better: 3.56%.** That is worth pausing on, because it is the opposite
of what people expect a log transform to do.

The transform cut the skew from 3.69 to 1.12 - it genuinely compressed the tail. What it did not do
is merge the two populations. The festival days are still a separate cluster, still far from the
combined mean in log space, and still 3% of the rows. **A transform reshapes a distribution; it
cannot make two processes into one.**

**The generalisable check:** before applying any normality-based rule, compute what share of the
data it would flag. If it is far from 0.3%, the assumption has already failed and the rule is
arbitrary rather than principled - and no amount of transforming will rescue it if the cause is a
second population rather than a heavy tail.

## E7 · A quantile rule instead

In [ ]:
low, high = np.quantile(rentals, [0.01, 0.99])
quantile_keep = (rentals >= low) & (rentals <= high)
sigma_keep = (np.abs(rentals - rentals.mean()) <= 3 * rentals.std())

print(f"quantile rule removes {(~quantile_keep).sum()} rows, "
      f"{(festival & ~quantile_keep).sum()} of {festival.sum()} festivals")
print(f"three-sigma removes   {(~sigma_keep).sum()} rows, "
      f"{(festival & ~sigma_keep).sum()} of {festival.sum()} festivals\n")

X, y = days[["temp_c", "festival"]].to_numpy(), rentals

def evaluate(keep, label):
    model = LinearRegression().fit(X[train & keep], y[train & keep])
    p = model.predict(X[test])
    print(f"{label:<28} all {mean_absolute_error(y[test], p):6.1f} | "
          f"ordinary {mean_absolute_error(y[test][~is_festival_test], p[~is_festival_test]):5.1f} | "
          f"festival {mean_absolute_error(y[test][is_festival_test], p[is_festival_test]):7.1f}")

evaluate(np.ones(n_days, dtype=bool), "keep everything")
evaluate(quantile_keep, "quantile (1st-99th pct)")
evaluate(sigma_keep, "three-sigma")

**The quantile rule does far less damage: festival error 9.1 against three-sigma's 258.4.**

**Why.** A quantile rule removes a *fixed fraction* - 1% at each end, 18 rows - by construction. It
cannot delete an entire 3% population, so 20 of the 29 festivals survive in the data, the model
still learns that `festival = 1` means "much busier", and its festival predictions stay good.

**But do not conclude that quantile rules are safe.** They rescued this dataset by an accident of
proportions: festivals are 3.2% of days and the rule trims 2%. Had festivals been 0.5% of days, a
1%-each-end trim would have removed every one of them and produced exactly the three-sigma disaster.

**The honest conclusion:** the quantile rule is *better* than three-sigma because it makes no
distributional assumption and its damage is bounded. It is still a rule that deletes rows on the
basis of their value alone, which cannot distinguish an error from a different population. The
reason it worked here is arithmetic, not understanding.

**Notice also that "keep everything" is still the best row in the table.** The best treatment of
these outliers was not a better removal rule. It was not removing them.

## E8 · "19.5 is fine, we're within 20 bikes"

The sentence treats an average as though it described every case, and this average describes almost
none of them.

**What the 19.5 actually is:** a blend of 192 ordinary days at 9.6 and 8 festival days at 258.4. No
day in the test set had an error near 19.5. It is the arithmetic centre of two very different
populations - the same complaint the mean of a skewed distribution attracted earlier in this chapter,
now applied to the errors instead of the data.

**Why it matters for the use:** the model exists so Maria knows how many bikes to bring out. On an
ordinary day, being 10 bikes out means a few spare or a few customers waiting - cheap. On a festival
day, being 250 short means turning away hundreds of people on the day the stand could have earned
most, and there are only a handful of such days a year. **The error is concentrated exactly where
the cost per error is highest.**

**The right sentence:** "the model is accurate to about 10 bikes on ordinary days and has no idea
festivals exist - on those it is short by around 250." That is two numbers and it makes the decision
obvious, where one number hid it.

**The habit:** never accept an aggregate error without asking how it distributes across the cases
that matter. That is 07-05.

## E9 · Good on most customers, bad on the largest accounts

Three explanations from this chapter, in the order I would test them:

1. **The loss function is dominated by the many small accounts.** Fitting to minimise average
   squared or absolute error on a heavily skewed revenue distribution means the model optimises
   almost entirely for the numerous small customers - they are most of the rows. The large accounts
   are a handful of rows and contribute little to the objective, however much revenue they
   represent. **Check:** the share of rows against the share of revenue held by the top accounts. If
   2% of rows hold 60% of revenue, the model is optimising the wrong thing by construction.
2. **The large accounts are a different population.** Enterprise customers may be driven by contract
   cycles, procurement and account management, while small ones respond to price and seasonality. One
   model fitted to both learns the small-account relationship and applies it where it does not hold.
   **Check:** fit separately and compare - if the coefficients differ in sign or magnitude, they are
   different processes.
3. **Someone removed them.** A three-sigma or top-percentile filter somewhere in the pipeline,
   applied to revenue, deletes precisely the largest accounts. **Check:** the maximum revenue in the
   training data against the maximum in the raw data.

**What I would look at first: the third**, because it is a two-line check and it is a complete
explanation if true. Then the first, which is one `groupby`.

**And the framing to bring to whoever asked:** an aggregate error is the wrong metric for this
business. If 2% of customers are 60% of revenue, the model should be evaluated on
**revenue-weighted** error, or separately by segment. Optimising unweighted error on a skewed target
means being accurate about the customers who matter least. That is a metric-choice problem (05-04),
and it looks like an outlier problem.

## E10 · "How do you handle outliers?"

> First I look at them, because "outlier" is a question rather than a category. I want to know
> whether the value is impossible, in which case it is an error and I correct it or set it to
> missing; or possible and rare, in which case it is evidence and I keep it; or produced by a
> different process, in which case it is a second population and I keep it and add a feature saying
> so. Thresholds like three standard deviations I avoid as a default, because they assume normality
> and the standard deviation is inflated by the very points being tested - on skewed data that either
> deletes an entire real population or misses the extreme entirely, and I have seen both. Whatever I
> do, I check the error by segment afterwards rather than an overall average, because the rows with
> the largest errors are usually the rows that matter most.

**What is being assessed:** whether you reach for a rule or for the rows. Any specific threshold as
a first answer signals someone who has not been burned by one.

## E11 · A weekly number and a warning system

**The typical-day number: the median, reported alongside the interquartile range.**

> "A typical day this week: 104 rentals (middle half of days: 88 to 120)."

The median because festivals must not move it, and the range because a single number cannot
distinguish a steady week from a wild one. Two numbers, one line, no way to mislead.

**The warning system: a quantile threshold on the recent past, with the reason attached.**

Flag a day when it falls above the 95th or below the 5th percentile of the *last 90 days*. Three
deliberate choices:

- **Quantiles, not sigmas** - no normality assumption, and the threshold cannot be inflated by the
  events it is meant to catch (E4's masking).
- **A rolling window** - so the definition of "unusual" tracks the season. A busy day in February is
  not a busy day in July.
- **Both tails** - an unusually *quiet* day is a broken counter, a closure or bad weather, and is
  just as worth knowing about.

**What the warning should say** matters as much as when it fires:

> "Tuesday: 372 rentals, above the 95th percentile of the last 90 days (upper threshold 190). The
> festival flag was set."

Value, threshold, comparison period, and whether anything known explains it. A warning that says
only "anomaly detected" makes the reader do all the work and gets ignored within a fortnight.

**And the thing the design must not do:** feed the flagged days back as rows to be excluded from the
forecast. The warning is for Maria's attention, not for the training set.

## E12 · Five outliers in the wild

| | Probable kind | What I would do |
|---|---|---|
| (a) A patient recorded as 700 kg | **Error** - almost certainly a unit or decimal slip (70.0 kg) | Check against height and the recording system. Set to missing rather than guessing; a wrong weight drives a wrong drug dose |
| (b) A 40,000 EUR transaction where the average is 60 | **Rare but real, or a different population** | Do not delete. It is either a genuine large purchase - the most valuable row you have - or a business account behaving differently. Find out which, and add a feature if it is a segment |
| (c) A 30-second response when the median is 80 ms | **Rare but real, and the point of the data** | Keep it. In latency work the tail *is* the subject - users experience the slow requests, not the median. Report percentiles, never a mean |
| (d) A house price of 1 EUR | **Error, or a different process** | Not a market transaction: a transfer between family members, a nominal sale, a data-entry placeholder. Exclude it from a market-price model and record why |
| (e) A sensor reading of exactly 0.00 on 4% of days | **A sentinel, not an outlier at all** | The repetition and the exactness are the tell (02-04). It means "no reading". Convert to missing and add an indicator |

**(c) is the one that reverses the instinct.** In most contexts a value 375 times the median is
suspicious. In latency it is the whole business: 99th-percentile response time is what users
complain about, and a team that removed those rows would be optimising an experience nobody has.

**(e) is the trap** - it is not an outlier question at all, and treating it as one leads to
"removing" 4% of days rather than recognising that those days have no measurement.

## E13 · Explaining it to Maria

> The tidy-up rule looked at how far each day was from average and threw away anything unusually
> far. Your festival days are unusually far - that is what makes them festivals. So the forecast
> learned from a year with no festivals in it, and now it thinks every day is an ordinary Tuesday.
> The busiest days are the ones you most need it to get right.

(64 words.)

**The sentence doing the work is the second one.** "Your festival days are unusually far - that is
what makes them festivals" turns the rule's criterion into an obvious absurdity without any
statistics. Once someone sees that the filter's definition of "weird" is identical to the thing they
care about, no further argument is needed.

## E14 · Four treatments, compared honestly

In [ ]:
X_flag = days[["temp_c", "festival"]].to_numpy()
X_noflag = days[["temp_c"]].to_numpy()

def report(predicted, label):
    print(f"{label:<32} all {mean_absolute_error(y[test], predicted):6.1f} | "
          f"ordinary {mean_absolute_error(y[test][~is_festival_test], predicted[~is_festival_test]):5.1f} | "
          f"festival {mean_absolute_error(y[test][is_festival_test], predicted[is_festival_test]):7.1f}")

report(LinearRegression().fit(X_flag[train], y[train]).predict(X_flag[test]),
       "keep everything, WITH flag")
report(LinearRegression().fit(X_noflag[train], y[train]).predict(X_noflag[test]),
       "keep everything, no flag")
report(LinearRegression().fit(X_flag[train & sigma_keep], y[train & sigma_keep]).predict(X_flag[test]),
       "three-sigma removal")

ordinary_model = LinearRegression().fit(X_noflag[train & ~festival], y[train & ~festival])
festival_model = LinearRegression().fit(X_noflag[train & festival], y[train & festival])
split_prediction = np.where(festival[test],
                            festival_model.predict(X_noflag[test]),
                            ordinary_model.predict(X_noflag[test]))
report(split_prediction, "two separate models")

| Treatment | All days | Ordinary | Festival |
|---|---|---|---|
| Keep everything, **with flag** | **9.5** | 9.6 | **8.3** |
| Keep everything, **no flag** | 20.0 | 10.4 | 250.4 |
| Three-sigma removal | 19.5 | 9.6 | 258.4 |
| Two separate models | 9.6 | 9.6 | 9.1 |

**The finding that reframes the chapter: keeping the rows without the flag (250.4) is almost exactly
as bad as deleting them (258.4).**

So the damage was never really about the rows. It was about the **information**. A model that has the
festival days but no way to tell them apart is in the same position as a model that never saw them -
it fits one line through two populations and lands between them. Worse, its ordinary-day error is
slightly *higher* (10.4 against 9.6), because the unexplained festivals pull the fitted line upward.

**What I would ship: keep everything, with the flag.** It is the best on every segment, it is one
model, and it degrades gracefully - if a festival flag is ever missing at prediction time, the model
still produces a sensible ordinary-day estimate.

**Why not two separate models**, when they are nearly as good? Because they cost twice the
maintenance, need a routing rule, and the festival model is fitted on 21 training days - so it will
be unstable and cannot borrow the temperature relationship that both populations share. Separate
models earn their place when the groups differ in *which features matter*, not merely in level. Here
the temperature effect is common to both, and a shared model with an indicator captures that with
one extra column.

**The transferable rule:**

> When you find a second population, the question is not whether to remove it. It is **whether you
> can identify it** - and if you can, the answer is almost always a feature rather than a filter.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **02-06 · Univariate, bivariate,
multivariate: looking without fooling yourself**.